# Activation Oracle explanations for DIRECTER PICO activations

Reads residual-stream bundles from `activations_steered/` and `activations_unsteered/`,
injects them into the pretrained Qwen3-8B Activation Oracle, and records natural-language
explanations.

Injection follows the paper exactly: additive, norm-matched, at oracle block 1,
placeholder token `" ?"`, prefix `Layer: {source_layer}\n<placeholders>\n`.

Phases: inventory → integrity gates → redundancy collapse → model load →
plumbing check → null baselines → main sweep → aggregation.

In [3]:
"""Activation Oracle Explanations for A01-A08 (Task Instructions)"""

import json, os, time, contextlib
from pathlib import Path
from typing import Dict, List, Optional

import torch
import torch.nn.functional as F

torch.set_grad_enabled(False)

# --- Paths ---
ACT_DIR    = "/home/srinjoy/Anarva/oracle/activations_unsteered"
OUTPUT_DIR = "/home/srinjoy/Anarva/oracle/instruction_explanations"
STATUS     = "unsteered"

# --- Models ---
BASE_MODEL     = "Qwen/Qwen3-8B"
ORACLE_ADAPTER = "adamkarvonen/checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B"

# --- Scope ---
LAYER = 18  # 50% depth
ELEMENTS = ["p", "i", "c", "o"]

# --- Injection Math ---
INJECTION_LAYER = 1
STEERING_COEFFICIENT = 1.0
MAX_NEW_TOKENS = 60
BATCH_SIZE = 64

# Your exact requested questions
QUESTIONS = {
    "Q1": "What task is this activation associated with?",
    "Q2": "Which PICO element does this activation represent?",
    "Q3": "What semantic information is most strongly represented here?",
    "Q4": "What semantic concept or task does this activation represent?",
    "Q5": "What does this activation suggest the model is trying to do here?",
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Targeting Layer {LAYER} | Status: {STATUS} | Questions: {len(QUESTIONS)}")
print(f"CUDA Available: {torch.cuda.is_available()}")

Targeting Layer 18 | Status: unsteered | Questions: 5
CUDA Available: True


In [4]:
"""Load Qwen3-8B base + the pretrained Activation Oracle LoRA."""

from peft import LoraConfig, PeftConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers

print(f"transformers {transformers.__version__} | torch {torch.__version__}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "left"          
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

_sid = tokenizer.encode(" ?", add_special_tokens=False)
assert len(_sid) == 1, f"' ?' is not a single token: {_sid}"

_maj, _min = (int(x) for x in transformers.__version__.split(".")[:2])
_dtype_kw = "dtype" if (_maj, _min) >= (4, 56) else "torch_dtype"

print("Loading base model (bfloat16, cuda:0)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map={"": 0}, **{_dtype_kw: torch.bfloat16}
)
base_model.eval()

print("Fetching Oracle config to match target modules...")
oracle_config = PeftConfig.from_pretrained(ORACLE_ADAPTER)
dummy_config = LoraConfig(target_modules=oracle_config.target_modules)

print("Adding dummy adapter...")
base_model.add_adapter(dummy_config, adapter_name="default")

print("Loading Oracle LoRA adapter...")
model = PeftModel.from_pretrained(
    base_model, 
    ORACLE_ADAPTER, 
    adapter_name="activation_oracle"
)
model.eval()

# Confirm the adapter carries trained weights
nonzero = sum(1 for n, p in model.named_parameters()
              if "activation_oracle" in n and "lora_B" in n
              and p.detach().float().norm() > 0)

assert nonzero > 0, "Oracle adapter did not load!"
print(f"Oracle successfully loaded! Nonzero matrices: {nonzero} | dtype: {model.dtype}")

INPUT_DEVICE = model.get_input_embeddings().weight.device

/home/srinjoy/miniconda3/envs/qwenfix/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers 4.57.6 | torch 2.6.0+cu124
Loading base model (bfloat16, cuda:0)...


Loading checkpoint shards: 100%|██████████| 5/5 [00:03<00:00,  1.57it/s]


Fetching Oracle config to match target modules...


/home/srinjoy/miniconda3/envs/qwenfix/lib/python3.10/site-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['qalora_group_size', 'target_parameters', 'use_qalora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Adding dummy adapter...
Loading Oracle LoRA adapter...


/home/srinjoy/miniconda3/envs/qwenfix/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Oracle successfully loaded! Nonzero matrices: 252 | dtype: torch.bfloat16


In [5]:
"""Prompt format and residual-stream injection, per the Activation Oracles paper."""

SPECIAL_TOKEN = " ?"

def get_introspection_prefix(source_layer: int, num_positions: int) -> str:
    """Verbatim from the reference. The header and spacing are load-bearing."""
    return f"Layer: {source_layer}\n" + SPECIAL_TOKEN * num_positions + " \n"

def find_placeholders(token_ids, num_positions):
    """Locate the contiguous run of ' ?' placeholders in the tokenized prompt."""
    sid = tokenizer.encode(SPECIAL_TOKEN, add_special_tokens=False)[0]
    pos = [i for i, t in enumerate(token_ids) if t == sid][:num_positions]
    assert len(pos) == num_positions, f"found {len(pos)}/{num_positions} placeholders"
    return pos

def get_injection_submodule(layer: int):
    """The decoder block whose OUTPUT is modified. Always block 1."""
    for path in ("model.layers", "model.model.layers", "base_model.model.model.layers"):
        obj = model
        try:
            for part in path.split("."):
                obj = getattr(obj, part)
        except AttributeError:
            continue
        if isinstance(obj, torch.nn.ModuleList) and len(obj) > layer:
            return obj[layer]
    raise AttributeError("Could not locate the oracle's decoder layers.")

@contextlib.contextmanager
def temporary_hook(module, hook):
    h = module.register_forward_hook(hook)
    try:
        yield
    finally:
        h.remove()

def make_injection_hook(vectors, positions, coeff=1.0):
    """Paper Eq. 1:  h'_i = h_i + ||h_i|| * v_i / ||v_i||"""
    normed = [F.normalize(v, dim=-1).detach() for v in vectors]

    def hook_fn(module, _inp, output):
        if isinstance(output, tuple):
            resid, *rest = output; is_tuple = True
        else:
            resid, rest, is_tuple = output, [], False

        B, L, _ = resid.shape
        if L <= 1:                       
            return (resid, *rest) if is_tuple else resid

        for b in range(B):
            pos = torch.tensor(positions[b], dtype=torch.long, device=resid.device)
            if int(pos.max()) >= L:
                continue
            orig = resid[b, pos, :]
            inj = normed[b].to(resid.device) * orig.norm(dim=-1, keepdim=True) * coeff
            resid[b, pos, :] = orig + inj.to(resid.dtype).detach()

        return (resid, *rest) if is_tuple else resid
    return hook_fn

INJECTION_SUBMODULE = get_injection_submodule(INJECTION_LAYER)
print(f"Injection block configured at Layer {INJECTION_LAYER}.")

Injection block configured at Layer 1.


In [7]:
"""Collect the residual stream sitting on the A01-A08 instruction words."""

def load_instruction_spans():
    """Loads A01-A08 tokens (e.g., 'Participant', '(P)') from Document 1."""
    d = Path(ACT_DIR)
    records = []
    
    # Map elements to their specific instruction probes
    probe_map = {
        "p": ["A01", "A02"], # Participant, (P)
        "i": ["A03", "A04"], # Intervention, (I)
        "c": ["A05", "A06"], # Control, (C)
        "o": ["A07", "A08"]  # Outcome, (O)
    }

    # We only need document "1" because instructions are identical across documents
    doc = "1" 
    
    for element in ELEMENTS:
        bundle_path = d / f"{doc}_{STATUS}_L{LAYER}_{element}.pt"
        if not bundle_path.exists():
            print(f"Warning: Could not find {bundle_path.name}")
            continue

        payload = torch.load(bundle_path, map_location="cpu", weights_only=False)
        acts = payload["activations"]
        text_meta = payload["metadata"].get("activation_text", {})

        for probe_id in probe_map[element]:
            if probe_id in acts and acts[probe_id] is not None:
                vec = acts[probe_id].float()
                
                # Ensure shape is [1, hidden] for the Oracle
                if vec.ndim == 1:
                    vec = vec.unsqueeze(0)
                    
                span_text = text_meta.get(probe_id, "[UNKNOWN]")

                records.append({
                    "doc": doc, 
                    "element": element, 
                    "probe_id": probe_id,
                    "span_text": span_text, 
                    "vectors": vec
                })

    print(f"Successfully loaded {len(records)} instruction probes (A01-A08).")
    print("-" * 50)
    for r in records:
        print(f"[{r['element'].upper()}] Probe {r['probe_id']}: {r['span_text']!r}")
        
    return records

SPANS = load_instruction_spans()

Successfully loaded 8 instruction probes (A01-A08).
--------------------------------------------------
[P] Probe A01: 'Participant'
[P] Probe A02: '(P)'
[I] Probe A03: 'Intervention'
[I] Probe A04: '(I)'
[C] Probe A05: 'Control'
[C] Probe A06: '(C)'
[O] Probe A07: 'Outcome'
[O] Probe A08: '(O)'


In [8]:
"""Ask all six questions of the A01-A08 instruction spans."""

@torch.no_grad()
def run_batch(tasks):
    seqs, positions, vectors = [], [], []
    for t in tasks:
        v = t["vectors"]; K = v.shape[0]
        prompt = get_introspection_prefix(LAYER, K) + t["question"]
        ids = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}], tokenize=True,
            add_generation_prompt=True, return_tensors=None,
            padding=False, enable_thinking=False,
        )
        seqs.append(ids)
        positions.append(find_placeholders(ids, K))
        vectors.append(v)

    maxlen, pad = max(len(s) for s in seqs), tokenizer.pad_token_id
    ids_b, attn_b, pos_b = [], [], []
    for ids, pos in zip(seqs, positions):
        p = maxlen - len(ids)                       # left padding
        ids_b.append([pad] * p + list(ids))
        attn_b.append([0] * p + [1] * len(ids))
        pos_b.append([x + p for x in pos])          # positions move with the padding

    input_ids = torch.tensor(ids_b, dtype=torch.long, device=INPUT_DEVICE)
    attn = torch.tensor(attn_b, dtype=torch.long, device=INPUT_DEVICE)

    hook = make_injection_hook(vectors, pos_b, STEERING_COEFFICIENT)
    with temporary_hook(INJECTION_SUBMODULE, hook):
        out = model.generate(input_ids=input_ids, attention_mask=attn,
                             max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)

    answers = tokenizer.batch_decode(out[:, input_ids.shape[1]:], skip_special_tokens=True)
    
    # Store results safely
    return [{**{k: v for k, v in t.items() if k != "vectors"},
             "answer": a.strip()}
            for t, a in zip(tasks, answers)]

# Build the tasks
TASKS = []
for r in SPANS:
    for qid, q in QUESTIONS.items():
        TASKS.append({
            "element": r["element"],
            "probe_id": r["probe_id"],
            "span_text": r["span_text"],
            "qid": qid,
            "question": q,
            "vectors": r["vectors"]
        })

print(f"Running {len(TASKS)} queries through the Oracle...")

RESULTS = []
# Process in batches
for i in range(0, len(TASKS), BATCH_SIZE):
    chunk = TASKS[i:i + BATCH_SIZE]
    RESULTS.extend(run_batch(chunk))

print("Oracle execution complete!")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Running 40 queries through the Oracle...
Oracle execution complete!


In [9]:
"""Display the Oracle Explanations for A01-A08."""

print("=" * 90)
print("ORACLE EXPLANATIONS FOR TASK INSTRUCTIONS (A01-A08)")
print(f"Source Layer: {LAYER} | Status: {STATUS}")
print("=" * 90)

# Group and print by Probe ID
for probe_id in sorted(list(set(r["probe_id"] for r in RESULTS))):
    # Get basic details for this probe
    sample = [r for r in RESULTS if r["probe_id"] == probe_id][0]
    
    print(f"\n[{sample['element'].upper()}] PROBE {probe_id} | Text: {sample['span_text']!r}")
    print("-" * 90)
    
    for qid, q_text in QUESTIONS.items():
        # Find the specific answer
        ans = [r["answer"] for r in RESULTS if r["probe_id"] == probe_id and r["qid"] == qid][0]
        
        print(f"Q: {q_text}")
        print(f"A: {ans}\n")

ORACLE EXPLANATIONS FOR TASK INSTRUCTIONS (A01-A08)
Source Layer: 18 | Status: unsteered

[P] PROBE A01 | Text: 'Participant'
------------------------------------------------------------------------------------------
Q: What task is this activation associated with?
A: This activation is associated with the task of identifying and extracting the Participant entity from the text.

Q: Which PICO element does this activation represent?
A: The activation corresponds to the 'Outcome' element in the PICO framework.

Q: What semantic information is most strongly represented here?
A: The semantic information most strongly represented is the identification of participants in the events described.

Q: What semantic concept or task does this activation represent?
A: This activation represents the task of identifying and extracting the Participant entity from the text.

Q: What does this activation suggest the model is trying to do here?
A: The activation suggests the model is attempting to extract

# without question 3

In [10]:
"""Activation Oracle Explanations for A01-A08 (Task Instructions)"""

import json, os, time, contextlib
from pathlib import Path
from typing import Dict, List, Optional

import torch
import torch.nn.functional as F

torch.set_grad_enabled(False)

# --- Paths ---
ACT_DIR    = "/home/srinjoy/Anarva/oracle/activations_unsteered"
OUTPUT_DIR = "/home/srinjoy/Anarva/oracle/instruction_explanations"
STATUS     = "unsteered"

# --- Models ---
BASE_MODEL     = "Qwen/Qwen3-8B"
ORACLE_ADAPTER = "adamkarvonen/checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B"

# --- Scope ---
LAYER = 18  # 50% depth
ELEMENTS = ["p", "i", "c", "o"]

# --- Injection Math ---
INJECTION_LAYER = 1
STEERING_COEFFICIENT = 1.0
MAX_NEW_TOKENS = 60
BATCH_SIZE = 64

# Your exact requested questions
QUESTIONS = {
    "Q1": "What task is this activation associated with?",
    "Q2": "Which PICO element does this activation represent?",
    "Q3": "What semantic information is most strongly represented here?",
    "Q4": "What semantic concept or task does this activation represent?",
    "Q5": "What does this activation suggest the model is trying to do here?",
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Targeting Layer {LAYER} | Status: {STATUS} | Questions: {len(QUESTIONS)}")
print(f"CUDA Available: {torch.cuda.is_available()}")

Targeting Layer 18 | Status: unsteered | Questions: 5
CUDA Available: True


In [11]:
"""Load Qwen3-8B base + the pretrained Activation Oracle LoRA."""

from peft import LoraConfig, PeftConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers

print(f"transformers {transformers.__version__} | torch {torch.__version__}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "left"          
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

_sid = tokenizer.encode(" ?", add_special_tokens=False)
assert len(_sid) == 1, f"' ?' is not a single token: {_sid}"

_maj, _min = (int(x) for x in transformers.__version__.split(".")[:2])
_dtype_kw = "dtype" if (_maj, _min) >= (4, 56) else "torch_dtype"

print("Loading base model (bfloat16, cuda:0)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map={"": 0}, **{_dtype_kw: torch.bfloat16}
)
base_model.eval()

print("Fetching Oracle config to match target modules...")
oracle_config = PeftConfig.from_pretrained(ORACLE_ADAPTER)
dummy_config = LoraConfig(target_modules=oracle_config.target_modules)

print("Adding dummy adapter...")
base_model.add_adapter(dummy_config, adapter_name="default")

print("Loading Oracle LoRA adapter...")
model = PeftModel.from_pretrained(
    base_model, 
    ORACLE_ADAPTER, 
    adapter_name="activation_oracle"
)
model.eval()

# Confirm the adapter carries trained weights
nonzero = sum(1 for n, p in model.named_parameters()
              if "activation_oracle" in n and "lora_B" in n
              and p.detach().float().norm() > 0)

assert nonzero > 0, "Oracle adapter did not load!"
print(f"Oracle successfully loaded! Nonzero matrices: {nonzero} | dtype: {model.dtype}")

INPUT_DEVICE = model.get_input_embeddings().weight.device

transformers 4.57.6 | torch 2.6.0+cu124
Loading base model (bfloat16, cuda:0)...


Loading checkpoint shards: 100%|██████████| 5/5 [00:03<00:00,  1.53it/s]


Fetching Oracle config to match target modules...
Adding dummy adapter...
Loading Oracle LoRA adapter...
Oracle successfully loaded! Nonzero matrices: 252 | dtype: torch.bfloat16


In [12]:
"""Collect the residual stream sitting on the A01-A08 instruction words."""

def load_instruction_spans():
    """Loads A01-A08 tokens (e.g., 'Participant', '(P)') from Document 1."""
    d = Path(ACT_DIR)
    records = []
    
    # Map elements to their specific instruction probes
    probe_map = {
        "p": ["A01", "A02"], # Participant, (P)
        "i": ["A03", "A04"], # Intervention, (I)
        "c": ["A05", "A06"], # Control, (C)
        "o": ["A07", "A08"]  # Outcome, (O)
    }

    # We only need document "1" because instructions are identical across documents
    doc = "1" 
    
    for element in ELEMENTS:
        bundle_path = d / f"{doc}_{STATUS}_L{LAYER}_{element}.pt"
        if not bundle_path.exists():
            print(f"Warning: Could not find {bundle_path.name}")
            continue

        payload = torch.load(bundle_path, map_location="cpu", weights_only=False)
        acts = payload["activations"]
        text_meta = payload["metadata"].get("activation_text", {})

        for probe_id in probe_map[element]:
            if probe_id in acts and acts[probe_id] is not None:
                vec = acts[probe_id].float()
                
                # Ensure shape is [1, hidden] for the Oracle
                if vec.ndim == 1:
                    vec = vec.unsqueeze(0)
                    
                span_text = text_meta.get(probe_id, "[UNKNOWN]")

                records.append({
                    "doc": doc, 
                    "element": element, 
                    "probe_id": probe_id,
                    "span_text": span_text, 
                    "vectors": vec
                })

    print(f"Successfully loaded {len(records)} instruction probes (A01-A08).")
    print("-" * 50)
    for r in records:
        print(f"[{r['element'].upper()}] Probe {r['probe_id']}: {r['span_text']!r}")
        
    return records

SPANS = load_instruction_spans()

Successfully loaded 8 instruction probes (A01-A08).
--------------------------------------------------
[P] Probe A01: 'Participant'
[P] Probe A02: '(P)'
[I] Probe A03: 'Intervention'
[I] Probe A04: '(I)'
[C] Probe A05: 'Control'
[C] Probe A06: '(C)'
[O] Probe A07: 'Outcome'
[O] Probe A08: '(O)'


In [13]:
"""Prompt format and residual-stream injection, per the Activation Oracles paper."""

SPECIAL_TOKEN = " ?"

def get_introspection_prefix(source_layer: int, num_positions: int) -> str:
    """Verbatim from the reference. The header and spacing are load-bearing."""
    return f"Layer: {source_layer}\n" + SPECIAL_TOKEN * num_positions + " \n"

def find_placeholders(token_ids, num_positions):
    """Locate the contiguous run of ' ?' placeholders in the tokenized prompt."""
    sid = tokenizer.encode(SPECIAL_TOKEN, add_special_tokens=False)[0]
    pos = [i for i, t in enumerate(token_ids) if t == sid][:num_positions]
    assert len(pos) == num_positions, f"found {len(pos)}/{num_positions} placeholders"
    return pos

def get_injection_submodule(layer: int):
    """The decoder block whose OUTPUT is modified. Always block 1."""
    for path in ("model.layers", "model.model.layers", "base_model.model.model.layers"):
        obj = model
        try:
            for part in path.split("."):
                obj = getattr(obj, part)
        except AttributeError:
            continue
        if isinstance(obj, torch.nn.ModuleList) and len(obj) > layer:
            return obj[layer]
    raise AttributeError("Could not locate the oracle's decoder layers.")

@contextlib.contextmanager
def temporary_hook(module, hook):
    h = module.register_forward_hook(hook)
    try:
        yield
    finally:
        h.remove()

def make_injection_hook(vectors, positions, coeff=1.0):
    """Paper Eq. 1:  h'_i = h_i + ||h_i|| * v_i / ||v_i||"""
    normed = [F.normalize(v, dim=-1).detach() for v in vectors]

    def hook_fn(module, _inp, output):
        if isinstance(output, tuple):
            resid, *rest = output; is_tuple = True
        else:
            resid, rest, is_tuple = output, [], False

        B, L, _ = resid.shape
        if L <= 1:                       
            return (resid, *rest) if is_tuple else resid

        for b in range(B):
            pos = torch.tensor(positions[b], dtype=torch.long, device=resid.device)
            if int(pos.max()) >= L:
                continue
            orig = resid[b, pos, :]
            inj = normed[b].to(resid.device) * orig.norm(dim=-1, keepdim=True) * coeff
            resid[b, pos, :] = orig + inj.to(resid.dtype).detach()

        return (resid, *rest) if is_tuple else resid
    return hook_fn

INJECTION_SUBMODULE = get_injection_submodule(INJECTION_LAYER)
print(f"Injection block configured at Layer {INJECTION_LAYER}.")

Injection block configured at Layer 1.


In [14]:
"""Ask all six questions of the A01-A08 instruction spans."""

@torch.no_grad()
def run_batch(tasks):
    seqs, positions, vectors = [], [], []
    for t in tasks:
        v = t["vectors"]; K = v.shape[0]
        prompt = get_introspection_prefix(LAYER, K) + t["question"]
        ids = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}], tokenize=True,
            add_generation_prompt=True, return_tensors=None,
            padding=False, enable_thinking=False,
        )
        seqs.append(ids)
        positions.append(find_placeholders(ids, K))
        vectors.append(v)

    maxlen, pad = max(len(s) for s in seqs), tokenizer.pad_token_id
    ids_b, attn_b, pos_b = [], [], []
    for ids, pos in zip(seqs, positions):
        p = maxlen - len(ids)                       # left padding
        ids_b.append([pad] * p + list(ids))
        attn_b.append([0] * p + [1] * len(ids))
        pos_b.append([x + p for x in pos])          # positions move with the padding

    input_ids = torch.tensor(ids_b, dtype=torch.long, device=INPUT_DEVICE)
    attn = torch.tensor(attn_b, dtype=torch.long, device=INPUT_DEVICE)

    hook = make_injection_hook(vectors, pos_b, STEERING_COEFFICIENT)
    with temporary_hook(INJECTION_SUBMODULE, hook):
        out = model.generate(input_ids=input_ids, attention_mask=attn,
                             max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)

    answers = tokenizer.batch_decode(out[:, input_ids.shape[1]:], skip_special_tokens=True)
    
    # Store results safely
    return [{**{k: v for k, v in t.items() if k != "vectors"},
             "answer": a.strip()}
            for t, a in zip(tasks, answers)]

# Build the tasks
TASKS = []
for r in SPANS:
    for qid, q in QUESTIONS.items():
        TASKS.append({
            "element": r["element"],
            "probe_id": r["probe_id"],
            "span_text": r["span_text"],
            "qid": qid,
            "question": q,
            "vectors": r["vectors"]
        })

print(f"Running {len(TASKS)} queries through the Oracle...")

RESULTS = []
# Process in batches
for i in range(0, len(TASKS), BATCH_SIZE):
    chunk = TASKS[i:i + BATCH_SIZE]
    RESULTS.extend(run_batch(chunk))

print("Oracle execution complete!")

Running 40 queries through the Oracle...
Oracle execution complete!


In [15]:
"""Display the Oracle Explanations for A01-A08."""

print("=" * 90)
print("ORACLE EXPLANATIONS FOR TASK INSTRUCTIONS (A01-A08)")
print(f"Source Layer: {LAYER} | Status: {STATUS}")
print("=" * 90)

# Group and print by Probe ID
for probe_id in sorted(list(set(r["probe_id"] for r in RESULTS))):
    # Get basic details for this probe
    sample = [r for r in RESULTS if r["probe_id"] == probe_id][0]
    
    print(f"\n[{sample['element'].upper()}] PROBE {probe_id} | Text: {sample['span_text']!r}")
    print("-" * 90)
    
    for qid, q_text in QUESTIONS.items():
        # Find the specific answer
        ans = [r["answer"] for r in RESULTS if r["probe_id"] == probe_id and r["qid"] == qid][0]
        
        print(f"Q: {q_text}")
        print(f"A: {ans}\n")

ORACLE EXPLANATIONS FOR TASK INSTRUCTIONS (A01-A08)
Source Layer: 18 | Status: unsteered

[P] PROBE A01 | Text: 'Participant'
------------------------------------------------------------------------------------------
Q: What task is this activation associated with?
A: This activation is associated with the task of identifying and extracting the Participant entity from the text.

Q: Which PICO element does this activation represent?
A: The activation corresponds to the 'Outcome' element in the PICO framework.

Q: What semantic information is most strongly represented here?
A: The semantic information most strongly represented is the identification of participants in the events described.

Q: What semantic concept or task does this activation represent?
A: This activation represents the task of identifying and extracting the Participant entity from the text.

Q: What does this activation suggest the model is trying to do here?
A: The activation suggests the model is attempting to extract

# A16 activation explanation

In [26]:
"""Collect the A16 extracted medical spans (Full-Span logic by Claude)."""
from pathlib import Path

MAX_DOCS = 3  # Limit to first 3 documents for a clean printout
USE_FULL_SPAN = True

def _abstract_offsets(abstract_ids):
    """Maps token indices to character positions to avoid BPE spacing bugs."""
    pieces = [tokenizer.decode([t]) for t in abstract_ids]
    offs, pos = [], 0
    for p in pieces:
        offs.append((pos, pos + len(p)))
        pos += len(p)
    return "".join(pieces), offs

def _span_token_range(text, offs, needle):
    """Finds the inclusive token range covering the extracted text."""
    i = text.find(needle.strip())
    if i == -1: return None
    j = i + len(needle.strip())
    idx = [t for t, (s, e) in enumerate(offs) if s < j and e > i]
    return (idx[0], idx[-1] + 1) if idx else None

def load_a16_spans():
    d = Path(ACT_DIR)
    records = []
    
    # Get all available documents and take the first few
    all_sidecars = sorted(d.glob(f"*_{STATUS}_L{LAYER}_*_a16.pt"))
    unique_docs = sorted(list(set(sc.name.split("_")[0] for sc in all_sidecars)), key=int)
    target_docs = unique_docs[:MAX_DOCS] if MAX_DOCS else unique_docs

    for doc in target_docs:
        for element in ELEMENTS:
            bundle_path = d / f"{doc}_{STATUS}_L{LAYER}_{element}.pt"
            sidecar_path = d / f"{doc}_{STATUS}_L{LAYER}_{element}_a16.pt"
            
            if not sidecar_path.exists() or not bundle_path.exists():
                continue
                
            side = torch.load(sidecar_path, map_location="cpu", weights_only=False)
            texts = side["metadata"]["activation_text"]["A16"]
            a16_single = side["activations"]["A16"]

            if USE_FULL_SPAN:
                bundle = torch.load(bundle_path, map_location="cpu", weights_only=False)
                a12 = bundle["activations"]["A12"].float()
                meta = bundle["metadata"]
                s, e = meta["activation_token_spans"]["A12"]
                abstract_ids = meta["prompt_token_ids"][s:e]
                text, offs = _abstract_offsets(abstract_ids)

            for row, span_text in enumerate(texts):
                if USE_FULL_SPAN:
                    rng = _span_token_range(text, offs, span_text)
                    if rng is None or rng[1] > a12.shape[0]:
                        continue  # Could not find exact span
                    vec = a12[rng[0]:rng[1]].clone()  # [span_len, hidden]
                else:
                    vec = a16_single[row].float().unsqueeze(0)  # [1, hidden]

                records.append({
                    "doc": doc, "element": element, "span_row": row,
                    "span_text": span_text, "vectors": vec
                })

    print(f"Successfully loaded {len(records)} Extracted Medical Spans (A16) from {len(target_docs)} documents.")
    print("-" * 75)
    for r in records:
        print(f"[{r['element'].upper()}] Doc {r['doc']} | Tokens: {r['vectors'].shape[0]:2d} | Text: {r['span_text']!r}")
        
    return records

SPANS_A16 = load_a16_spans()

Successfully loaded 28 Extracted Medical Spans (A16) from 3 documents.
---------------------------------------------------------------------------
[P] Doc 1 | Tokens: 21 | Text: 'Patients with asthma stabilized on CFC albuterol during a 12-week safety and efficacy trial'
[I] Doc 1 | Tokens:  8 | Text: 'continue receiving CFC albuterol'
[I] Doc 1 | Tokens:  9 | Text: 'switched to receive HFA albuterol'
[I] Doc 1 | Tokens:  7 | Text: 'self-administered the study drug'
[C] Doc 1 | Tokens:  6 | Text: 'CFC albuterol'
[C] Doc 1 | Tokens:  6 | Text: 'CFC albuterol'
[C] Doc 1 | Tokens:  6 | Text: 'CFC albuterol'
[C] Doc 1 | Tokens:  6 | Text: 'HFA albuterol'
[C] Doc 1 | Tokens:  6 | Text: 'HFA albuterol'
[O] Doc 1 | Tokens: 12 | Text: 'Bronchodilator efficacy was evaluated by serial spirometry'
[O] Doc 1 | Tokens: 23 | Text: 'Safety was assessed by measuring changes in pulse rate, blood pressure, and electrocardiogram (ECG) intervals'
[O] Doc 1 | Tokens:  3 | Text: 'monitoring adverse events'


In [27]:
"""Ask all six questions of the A16 extracted spans."""

# Build the tasks
TASKS_A16 = []
for r in SPANS_A16:
    for qid, q in QUESTIONS.items():
        TASKS_A16.append({
            "doc": r["doc"],
            "element": r["element"],
            "span_text": r["span_text"],
            "qid": qid,
            "question": q,
            "vectors": r["vectors"]
        })

print(f"Running {len(TASKS_A16)} A16 queries through the Oracle...")

RESULTS_A16 = []
for i in range(0, len(TASKS_A16), BATCH_SIZE):
    chunk = TASKS_A16[i:i + BATCH_SIZE]
    RESULTS_A16.extend(run_batch(chunk))

print("Oracle execution complete!")

Running 140 A16 queries through the Oracle...
Oracle execution complete!


In [28]:
"""Display the Oracle Explanations for the Actual Extracted Phrases (A16)."""

print("=" * 90)
print("ORACLE EXPLANATIONS FOR EXTRACTED PICO SPANS (A16)")
print(f"Source Layer: {LAYER} | Status: {STATUS} | Full Span: {USE_FULL_SPAN}")
print("=" * 90)

# Group and print by Document and Element
grouped = defaultdict(list)
for r in RESULTS_A16:
    grouped[(r["doc"], r["element"], r["span_text"])].append(r)

for (doc, element, span_text), answers in grouped.items():
    print(f"\n[{element.upper()}] DOC {doc} | Extracted Text: {span_text!r}")
    print("-" * 90)
    
    for ans_dict in answers:
        print(f"Q: {ans_dict['question']}")
        print(f"A: {ans_dict['answer']}\n")

ORACLE EXPLANATIONS FOR EXTRACTED PICO SPANS (A16)
Source Layer: 18 | Status: unsteered | Full Span: True

[P] DOC 1 | Extracted Text: 'Patients with asthma stabilized on CFC albuterol during a 12-week safety and efficacy trial'
------------------------------------------------------------------------------------------
Q: What task is this activation associated with?
A: This activation is associated with the task of identifying the patient population in a clinical trial.

Q: Which PICO element does this activation represent?
A: This activation represents the 'Population' element in the PICO framework.

Q: What semantic information is most strongly represented here?
A: The semantic information most strongly represented is the patient population, which includes individuals with asthma stabilized on CFC albuterol.

Q: What semantic concept or task does this activation represent?
A: This activation represents the task of identifying and extracting the subject of the study from the text.

Q:

# explaining the <|im_start|>assistant\n<think>\n\n</think>\n\n token

In [18]:
"""See the A09 token and build the explanation tasks."""

from pathlib import Path
import torch

MAX_DOCS = 3  # Limit to 3 documents to keep the output readable

# Your exact questions
A09_QUESTIONS = {
    "Q1": "What task has the assistant been asked to perform?",
    "Q2": "What instructions is the assistant following?",
    "Q3": "What is the assistant about to do?",
    "Q4": "Can you describe what is happening in this text?",
    "Q5": "What are the PICO elements in the given abstract?"
}

def load_a09_tasks():
    d = Path(ACT_DIR)
    
    # Find all main bundles (ignore the _a16 sidecars)
    all_bundles = sorted(d.glob(f"*_{STATUS}_L{LAYER}_*.pt"))
    main_bundles = [f for f in all_bundles if "_a16" not in f.name]
    
    unique_docs = sorted(list(set(f.name.split("_")[0] for f in main_bundles)), key=int)
    target_docs = unique_docs[:MAX_DOCS] if MAX_DOCS else unique_docs

    tasks = []
    print(f"--- VERIFYING THE A09 TOKEN ---")
    
    for doc in target_docs:
        for element in ELEMENTS:
            bundle_path = d / f"{doc}_{STATUS}_L{LAYER}_{element}.pt"
            if not bundle_path.exists():
                continue

            payload = torch.load(bundle_path, map_location="cpu", weights_only=False)
            acts = payload["activations"]
            meta = payload["metadata"]
            
            if "A09" in acts and acts["A09"] is not None:
                # 1. SHOW THE TOKEN TEXT
                token_text = meta.get("activation_text", {}).get("A09", "UNKNOWN")
                print(f"[Doc {doc} | Element {element.upper()}] A09 corresponds to token: {token_text!r}")
                
                # 2. PREPARE THE VECTOR
                vec = acts["A09"].float()
                if vec.ndim == 1:
                    vec = vec.unsqueeze(0)

                for qid, q in A09_QUESTIONS.items():
                    tasks.append({
                        "doc": doc,
                        "element": element,
                        "qid": qid,
                        "question": q,
                        "vectors": vec
                    })
                    
    return tasks

TASKS_A09 = load_a09_tasks()
print(f"\nGenerated {len(TASKS_A09)} queries for the Oracle.")

--- VERIFYING THE A09 TOKEN ---
[Doc 1 | Element P] A09 corresponds to token: '\n\n'
[Doc 1 | Element I] A09 corresponds to token: '\n\n'
[Doc 1 | Element C] A09 corresponds to token: '\n\n'
[Doc 1 | Element O] A09 corresponds to token: '\n\n'
[Doc 2 | Element P] A09 corresponds to token: '\n\n'
[Doc 2 | Element I] A09 corresponds to token: '\n\n'
[Doc 2 | Element C] A09 corresponds to token: '\n\n'
[Doc 2 | Element O] A09 corresponds to token: '\n\n'
[Doc 3 | Element P] A09 corresponds to token: '\n\n'
[Doc 3 | Element I] A09 corresponds to token: '\n\n'
[Doc 3 | Element C] A09 corresponds to token: '\n\n'
[Doc 3 | Element O] A09 corresponds to token: '\n\n'

Generated 60 queries for the Oracle.


In [19]:
"""Run the Oracle on A09 and display the results cleanly."""

from collections import defaultdict

# 1. Run the Oracle
RESULTS_A09 = []
print("Running Oracle explanations...")
for i in range(0, len(TASKS_A09), BATCH_SIZE):
    chunk = TASKS_A09[i:i + BATCH_SIZE]
    RESULTS_A09.extend(run_batch(chunk))

# 2. Display the Results
print("\n" + "=" * 90)
print("ORACLE EXPLANATIONS FOR A09 (PRE-GENERATION INTENT STATE)")
print(f"Source Layer: {LAYER} | Status: {STATUS}")
print("=" * 90)

grouped = defaultdict(list)
for r in RESULTS_A09:
    grouped[(r["doc"], r["element"])].append(r)

for (doc, element), answers in grouped.items():
    print(f"\n[{element.upper()}] DOC {doc}")
    print("-" * 90)
    
    for ans_dict in answers:
        print(f"Q: {ans_dict['question']}")
        print(f"A: {ans_dict['answer']}\n")

Running Oracle explanations...

ORACLE EXPLANATIONS FOR A09 (PRE-GENERATION INTENT STATE)
Source Layer: 18 | Status: unsteered

[P] DOC 1
------------------------------------------------------------------------------------------
Q: What task has the assistant been asked to perform?
A: The assistant has been tasked with identifying and extracting the relevant information from the provided medical report.

Q: What instructions is the assistant following?
A: The assistant is following the instructions to extract and present only the relevant information from the provided text.

Q: What is the assistant about to do?
A: The assistant is about to provide a detailed analysis of the given sentence, focusing on the grammatical structure.

Q: Can you describe what is happening in this text?
A: The text describes a situation where a vehicle is moving forward at a speed of 10 km/h.

Q: What are the PICO elements in the given abstract?
A: The PICO elements in the abstract are: Population (P) - pati

In [21]:
"""Load A09 (The Assistant Start Token) and test with new Binary/Action questions."""

from pathlib import Path
from collections import defaultdict
import torch
import json

MAX_DOCS = 3  # Limit to 3 documents for a clean printout

# NEW QUESTIONS: Focused on Binary (Yes/No) and immediate assistant actions
# This matches the Oracle's CLS (Classification) and SPQA training data.
NEW_A09_QUESTIONS = {
    "Q1": "What the task the assistant is about to perform?",
    "Q2": "What instructions is the assistant following?",
    "Q3": "What is the assistant about to do?",
    "Q4": "What PICO element will the assistant extract/?",
    "Q5": "Is the assistant about to output a JSON object?",
    "Q6": "What is the assistant about to write?"
}

def load_and_run_new_a09():
    d = Path(ACT_DIR)
    all_bundles = sorted(d.glob(f"*_{STATUS}_L{LAYER}_*.pt"))
    main_bundles = [f for f in all_bundles if "_a16" not in f.name]
    
    unique_docs = sorted(list(set(f.name.split("_")[0] for f in main_bundles)), key=int)
    target_docs = unique_docs[:MAX_DOCS] if MAX_DOCS else unique_docs

    tasks = []
    print(f"Loading A09 from {len(target_docs)} documents...")
    
    for doc in target_docs:
        for element in ELEMENTS:
            bundle_path = d / f"{doc}_{STATUS}_L{LAYER}_{element}.pt"
            if not bundle_path.exists():
                continue

            payload = torch.load(bundle_path, map_location="cpu", weights_only=False)
            acts = payload["activations"]
            
            if "A09" in acts and acts["A09"] is not None:
                vec = acts["A09"].float()
                if vec.ndim == 1:
                    vec = vec.unsqueeze(0)

                for qid, q in NEW_A09_QUESTIONS.items():
                    tasks.append({
                        "doc": doc,
                        "element": element,
                        "qid": qid,
                        "question": q,
                        "vectors": vec
                    })
                    
    print(f"Running {len(tasks)} new queries through the Oracle...")
    
    results = []
    for i in range(0, len(tasks), BATCH_SIZE):
        chunk = tasks[i:i + BATCH_SIZE]
        results.extend(run_batch(chunk))
        
    return results

# Run the tasks
RESULTS_NEW_A09 = load_and_run_new_a09()

# Display the Results
print("\n" + "=" * 90)
print("ORACLE EXPLANATIONS FOR A09 (NEW BINARY/ACTION QUESTIONS)")
print(f"Source Layer: {LAYER} | Status: {STATUS}")
print("=" * 90)

grouped = defaultdict(list)
for r in RESULTS_NEW_A09:
    grouped[(r["doc"], r["element"])].append(r)

for (doc, element), answers in grouped.items():
    print(f"\n[{element.upper()}] DOC {doc}")
    print("-" * 90)
    
    for ans_dict in answers:
        print(f"Q: {ans_dict['question']}")
        print(f"A: {ans_dict['answer']}\n")

Loading A09 from 3 documents...
Running 72 new queries through the Oracle...

ORACLE EXPLANATIONS FOR A09 (NEW BINARY/ACTION QUESTIONS)
Source Layer: 18 | Status: unsteered

[P] DOC 1
------------------------------------------------------------------------------------------
Q: What the task the assistant is about to perform?
A: The assistant is about to perform the task of identifying and extracting specific data elements from a structured dataset.

Q: What instructions is the assistant following?
A: The assistant is following the instructions to extract and present only the relevant information from the provided text.

Q: What is the assistant about to do?
A: The assistant is about to provide a detailed analysis of the given sentence, focusing on the grammatical structure.

Q: What PICO element will the assistant extract/?
A: The assistant will extract the PICO element 'I' from the provided medical research paper.

Q: Is the assistant about to output a JSON object?
A: Yes, the assista

# explaining the a13 <|im_start|>assistant\n<think>\n\n</think>\n\n token but from the steered generation


In [23]:
"""Load A13 (The Steered Assistant Start Token) and test with Intent questions."""

from pathlib import Path
from collections import defaultdict
import torch

# --- Setup missing variables ---
STEERED_DIR = "/home/srinjoy/Anarva/oracle/activations_steered"
LAYER = 18
ELEMENTS = ["p", "i", "c", "o"]
BATCH_SIZE = 64
MAX_DOCS = 3  # Limit to 3 documents for a clean printout

# Your exact requested questions
A13_QUESTIONS = {
    "Q1": "What task is the assistant about to perform?",
    "Q2": "What instructions is the assistant following?",
    "Q3": "What is the assistant about to do?",
    "Q4": "What PICO element will the assistant extract?",
    "Q5": "Is the assistant about to output a JSON object?",
    "Q6": "What is the assistant about to write?"
}

def load_and_run_a13():
    d = Path(STEERED_DIR)
    all_bundles = sorted(d.glob(f"*_steered_L{LAYER}_*.pt"))
    main_bundles = [f for f in all_bundles if "_a16" not in f.name]
    
    unique_docs = sorted(list(set(f.name.split("_")[0] for f in main_bundles)), key=int)
    target_docs = unique_docs[:MAX_DOCS] if MAX_DOCS else unique_docs

    tasks = []
    print(f"Loading A13 (Steered Intent) from {len(target_docs)} documents...")
    print(f"--- VERIFYING THE A13 TOKEN ---")
    
    for doc in target_docs:
        for element in ELEMENTS:
            bundle_path = d / f"{doc}_steered_L{LAYER}_{element}.pt"
            if not bundle_path.exists():
                continue

            payload = torch.load(bundle_path, map_location="cpu", weights_only=False)
            acts = payload["activations"]
            meta = payload["metadata"]
            
            # Check if A13 exists 
            if "A13" in acts and acts["A13"] is not None:
                
                # SHOW THE EXACT TOKEN A13 SITS ON (It should be \n or \n\n)
                # (Note: Since A13 is the steered version of A09, it sits on the A09 token string)
                token_text = meta.get("activation_text", {}).get("A09", "UNKNOWN")
                print(f"[Doc {doc} | Element {element.upper()}] A13 corresponds to token: {token_text!r}")
                
                vec = acts["A13"].float()
                if vec.ndim == 1:
                    vec = vec.unsqueeze(0)

                for qid, q in A13_QUESTIONS.items():
                    tasks.append({
                        "doc": doc,
                        "element": element,
                        "qid": qid,
                        "question": q,
                        "vectors": vec
                    })
                    
    print(f"\nRunning {len(tasks)} queries through the Oracle...")
    
    results = []
    for i in range(0, len(tasks), BATCH_SIZE):
        chunk = tasks[i:i + BATCH_SIZE]
        results.extend(run_batch(chunk))  # Assumes run_batch is already in your notebook memory
        
    return results

# 1. Run the tasks
RESULTS_A13 = load_and_run_a13()

# 2. Display the Results cleanly
print("\n" + "=" * 90)
print("ORACLE EXPLANATIONS FOR A13 (STEERED INTENT STATE)")
print(f"Source Layer: {LAYER} | Status: STEERED")
print("=" * 90)

grouped = defaultdict(list)
for r in RESULTS_A13:
    grouped[(r["doc"], r["element"])].append(r)

if not grouped:
    print("WARNING: No A13 vectors found!")
else:
    for (doc, element), answers in grouped.items():
        print(f"\n[{element.upper()}] DOC {doc}")
        print("-" * 90)
        
        for ans_dict in answers:
            print(f"Q: {ans_dict['question']}")
            print(f"A: {ans_dict['answer']}\n")

Loading A13 (Steered Intent) from 3 documents...
--- VERIFYING THE A13 TOKEN ---
[Doc 1 | Element P] A13 corresponds to token: '\n\n'
[Doc 1 | Element I] A13 corresponds to token: '\n\n'
[Doc 1 | Element C] A13 corresponds to token: '\n\n'
[Doc 1 | Element O] A13 corresponds to token: '\n\n'
[Doc 2 | Element P] A13 corresponds to token: '\n\n'
[Doc 2 | Element I] A13 corresponds to token: '\n\n'
[Doc 2 | Element C] A13 corresponds to token: '\n\n'
[Doc 2 | Element O] A13 corresponds to token: '\n\n'
[Doc 3 | Element P] A13 corresponds to token: '\n\n'
[Doc 3 | Element I] A13 corresponds to token: '\n\n'
[Doc 3 | Element C] A13 corresponds to token: '\n\n'
[Doc 3 | Element O] A13 corresponds to token: '\n\n'

Running 72 queries through the Oracle...

ORACLE EXPLANATIONS FOR A13 (STEERED INTENT STATE)
Source Layer: 18 | Status: STEERED

[P] DOC 1
------------------------------------------------------------------------------------------
Q: What task is the assistant about to perform?
A: T

# explaining the A15 - very last token of your Steered Guideline block

In [24]:
"""Load A15 (Last Guideline Token) and verify the token string."""

from pathlib import Path
from collections import defaultdict
import torch

# --- Setup variables ---
ACT_DIR = "/home/srinjoy/Anarva/oracle/activations_unsteered"
LAYER = 18
ELEMENTS = ["p", "i", "c", "o"]
BATCH_SIZE = 64
MAX_DOCS = 3  # Limit to 3 documents for a clean printout

# Questions designed to test the model's understanding of the rules it just read.
A15_QUESTIONS = {
    "Q1": "What instructions is the assistant following?",
    "Q2": "What rules has the assistant been told to follow?",
    "Q3": "What PICO element do these instructions describe?",
    "Q4": "What task is the assistant about to perform?",
    "Q5": "Is this text a set of clinical guidelines?",
    "Q6": "What semantic concept is most strongly represented here?"
}

def load_and_visualize_a15():
    d = Path(ACT_DIR)
    all_bundles = sorted(d.glob(f"*_unsteered_L{LAYER}_*.pt"))
    main_bundles = [f for f in all_bundles if "_a16" not in f.name]
    
    unique_docs = sorted(list(set(f.name.split("_")[0] for f in main_bundles)), key=int)
    target_docs = unique_docs[:MAX_DOCS] if MAX_DOCS else unique_docs

    tasks = []
    print(f"Loading A15 (Guideline End) from {len(target_docs)} documents...")
    print(f"--- VERIFYING THE A15 TOKEN ---")
    
    for doc in target_docs:
        for element in ELEMENTS:
            bundle_path = d / f"{doc}_unsteered_L{LAYER}_{element}.pt"
            if not bundle_path.exists():
                continue

            payload = torch.load(bundle_path, map_location="cpu", weights_only=False)
            acts = payload["activations"]
            meta = payload["metadata"]
            
            if "A15" in acts and acts["A15"] is not None:
                # SHOW THE EXACT TOKEN A15 SITS ON
                # It should be the very last word/punctuation of the specific PICO guideline block
                token_text = meta.get("activation_text", {}).get("A15", "UNKNOWN")
                print(f"[Doc {doc} | Element {element.upper()}] A15 corresponds to token: {token_text!r}")
                
                vec = acts["A15"].float()
                if vec.ndim == 1:
                    vec = vec.unsqueeze(0)

                for qid, q in A15_QUESTIONS.items():
                    tasks.append({
                        "doc": doc,
                        "element": element,
                        "qid": qid,
                        "question": q,
                        "vectors": vec
                    })
                    
    print(f"\nGenerated {len(tasks)} queries for the Oracle.")
    return tasks

TASKS_A15 = load_and_visualize_a15()

Loading A15 (Guideline End) from 3 documents...
--- VERIFYING THE A15 TOKEN ---
[Doc 1 | Element P] A15 corresponds to token: '.\n'
[Doc 1 | Element I] A15 corresponds to token: '.\n'
[Doc 1 | Element C] A15 corresponds to token: '.\n'
[Doc 1 | Element O] A15 corresponds to token: '.\n'
[Doc 2 | Element P] A15 corresponds to token: '.\n'
[Doc 2 | Element I] A15 corresponds to token: '.\n'
[Doc 2 | Element C] A15 corresponds to token: '.\n'
[Doc 2 | Element O] A15 corresponds to token: '.\n'
[Doc 3 | Element P] A15 corresponds to token: '.\n'
[Doc 3 | Element I] A15 corresponds to token: '.\n'
[Doc 3 | Element C] A15 corresponds to token: '.\n'
[Doc 3 | Element O] A15 corresponds to token: '.\n'

Generated 72 queries for the Oracle.


In [25]:
"""Run the Oracle on A15 and display the explanations cleanly."""

# 1. Run the Oracle
RESULTS_A15 = []
print("Running Oracle explanations...")
for i in range(0, len(TASKS_A15), BATCH_SIZE):
    chunk = TASKS_A15[i:i + BATCH_SIZE]
    RESULTS_A15.extend(run_batch(chunk))  # Assumes run_batch is already defined in your notebook

# 2. Display the Results cleanly
print("\n" + "=" * 90)
print("ORACLE EXPLANATIONS FOR A15 (END OF GUIDELINES STATE)")
print(f"Source Layer: {LAYER} | Status: UNSTEERED")
print("=" * 90)

grouped = defaultdict(list)
for r in RESULTS_A15:
    grouped[(r["doc"], r["element"])].append(r)

if not grouped:
    print("WARNING: No A15 vectors found!")
else:
    for (doc, element), answers in grouped.items():
        print(f"\n[{element.upper()}] DOC {doc}")
        print("-" * 90)
        
        for ans_dict in answers:
            print(f"Q: {ans_dict['question']}")
            print(f"A: {ans_dict['answer']}\n")

Running Oracle explanations...

ORACLE EXPLANATIONS FOR A15 (END OF GUIDELINES STATE)
Source Layer: 18 | Status: UNSTEERED

[P] DOC 1
------------------------------------------------------------------------------------------
Q: What instructions is the assistant following?
A: The assistant is following instructions to extract and list all the nouns from the user's input.

Q: What rules has the assistant been told to follow?
A: The assistant is instructed to follow specific rules, such as identifying the subject and object, and to avoid including prepositions.

Q: What PICO element do these instructions describe?
A: The instructions describe the PICO element 'Population' by specifying the demographic group being studied.

Q: What task is the assistant about to perform?
A: The assistant is tasked with identifying and extracting the most relevant keywords from a given text.

Q: Is this text a set of clinical guidelines?
A: No, the text is a set of clinical guidelines for medical professio